# Netflix Big Data Analytics using PySpark

## Objective
The objective of this project is to analyze Netflix content data using PySpark and extract meaningful business insights from large-scale datasets efficiently.

## Technologies Used
- PySpark
- Python
- Matplotlib
- Pandas

## Dataset
Netflix Movies and TV Shows Dataset

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
import matplotlib.pyplot as plt
import pandas as pd
import time

from pyspark.sql.functions import explode, split, regexp_extract, rank
from pyspark import StorageLevel

# Spark Session Initialization

In [ ]:
spark = SparkSession.builder \
    .appName("Netflix Content Strategy Analytics") \
    .getOrCreate()
    
print("Spark Version:", spark.version)

# Dataset Loading

In [ ]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("multiLine", "true") \
    .option("escape", "\"") \
    .option("inferSchema", "false") \
    .load("Dataset.csv")

In [ ]:
print("Total Rows:", df.count())
print("Total Columns:", len(df.columns))

In [ ]:
df.printSchema()

In [ ]:
df.show(5)

In [ ]:
df= df.withColumn("release_year",col("release_year").cast("int"))

In [ ]:
null_counts = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
])

null_counts.show()

In [ ]:
df= df.fillna({"director":"Unknown","cast":"Unknown","country":"Unknown"})

In [ ]:
duplicate_count = df.count() - df.dropDuplicates().count()
print("Duplicate Records:", duplicate_count)

In [ ]:
df = df.dropDuplicates()

In [ ]:
df = df.withColumn(
    "duration_number",
    regexp_extract(col("duration"), "(\\d+)", 1).cast("int")
)

In [ ]:
df = df.withColumn(
    "duration_type",
    when(col("duration").contains("min"), "Movie")
    .otherwise("TV Show")
)

In [ ]:
df = df.withColumn(
    "year_added",
    regexp_extract(col("date_added"), "(\\d{4})", 1).cast("int")
)

In [ ]:
df = df.withColumn(
    "month_added",
    regexp_extract(
        col("date_added"),
        "([A-Za-z]+)",
        1
    )
)

In [ ]:
df = df.withColumn(
    "decade",
    concat(
        floor(col("release_year") / 10) * 10,
        lit("s")
    )
)

In [ ]:
movies_df = df.filter(col("type") == "Movie")

tvshows_df = df.filter(col("type") == "TV Show")

# Data Cleaning Summary

The preprocessing phase included:
- datatype conversion,
- null value handling,
- duplicate removal,
- duration extraction,
- temporal feature engineering,
- and content categorization.

These transformations improve downstream analytics quality and query efficiency.

In [ ]:
content_distribution = df.groupBy("type") \
    .count()

content_distribution.show()

## Insight

Movies dominate the Netflix catalog, indicating a stronger investment in film-based content compared to television series.

In [ ]:
country_df = df.withColumn(
    "country_single",
    explode(split(col("country"), ","))
)

country_analysis = country_df.groupBy(
    trim(col("country_single")).alias("country")
).count().orderBy(desc("count"))

country_analysis.show(10)

## Insight

The United States and India contribute significantly to Netflix content production, highlighting regional dominance in content distribution.

In [ ]:
genre_df = df.withColumn(
    "genre",
    explode(split(col("listed_in"), ","))
)

genre_analysis = genre_df.groupBy(
    trim(col("genre")).alias("genre")
).count().orderBy(desc("count"))

genre_analysis.show(10)

## Insight

Drama and International genres appear most frequently, suggesting strong audience demand for globally diverse storytelling.

In [ ]:
rating_analysis = df.groupBy("rating") \
    .count() \
    .orderBy(desc("count"))

rating_analysis.show()

In [ ]:
yearly_added = df.groupBy("year_added") \
    .count() \
    .orderBy("year_added")

yearly_added.show()

## Insight

Content additions accelerated significantly after 2015, reflecting Netflix’s aggressive global expansion strategy.

In [ ]:
movies_df = df.filter(col("type") == "Movie")

movies_df.select(
    avg("duration_number"),
    min("duration_number"),
    max("duration_number")
).show()

In [ ]:
tvshows_df = df.filter(col("type") == "TV Show")

tvshows_df.groupBy("duration_number") \
    .count() \
    .orderBy(desc("count")) \
    .show()

In [ ]:
director_analysis = df.filter(
    col("director") != "Unknown"
).groupBy("director") \
 .count() \
 .orderBy(desc("count"))

director_analysis.show(10)

# Advanced Analytics using Window Functions

Window functions are used to rank entities without collapsing the dataset structure, enabling advanced distributed analytics workflows.

In [ ]:
window_spec = Window.orderBy(desc("count"))

ranked_countries = country_analysis.withColumn(
    "rank",
    rank().over(window_spec)
)

ranked_countries.show(10)

# Distributed Performance Optimization

PySpark optimization techniques such as caching and repartitioning are applied to improve execution efficiency for repeated distributed operations.

In [ ]:
df.cache()

In [ ]:
print(df.rdd.getNumPartitions())

In [ ]:
partitioned_df = df.repartition(4)

print(partitioned_df.rdd.getNumPartitions())

In [ ]:
start = time.time()

benchmark = partitioned_df.groupBy(
    "type",
    "rating"
).count()

benchmark.show()

end = time.time()

processing_time = end - start

print(f"Processing Time: {processing_time:.3f} seconds")

In [ ]:
partitioned_df.groupBy(
    "country"
).count().explain()

## Observation

Repartitioning improves workload distribution across executors and enhances parallel processing performance during aggregation operations.

In [ ]:
yearly_pd = yearly_added.toPandas()

plt.figure(figsize=(12,5))

plt.plot(
    yearly_pd['year_added'],
    yearly_pd['count']
)

plt.title('Netflix Content Growth')
plt.xlabel('Year')
plt.ylabel('Titles Added')

plt.title("YOUR TITLE")
plt.xlabel("X LABEL")
plt.ylabel("Y LABEL")
plt.xticks(rotation=45)
plt.tight_layout()
plt.grid(True)
plt.show()

In [ ]:
country_pd = country_analysis.limit(10).toPandas()

plt.figure(figsize=(12,5))

plt.bar(
    country_pd['country'],
    country_pd['count']
)

plt.xticks(rotation=45)

plt.title('Top Countries on Netflix')

plt.title("YOUR TITLE")
plt.xlabel("X LABEL")
plt.ylabel("Y LABEL")
plt.xticks(rotation=45)
plt.tight_layout()
plt.grid(True)
plt.show()

In [ ]:
genre_pd = genre_analysis.limit(10).toPandas()

plt.figure(figsize=(12,5))

plt.bar(
    genre_pd['genre'],
    genre_pd['count']
)

plt.xticks(rotation=45)

plt.title('Top Netflix Genres')

plt.title("YOUR TITLE")
plt.xlabel("X LABEL")
plt.ylabel("Y LABEL")
plt.xticks(rotation=45)
plt.tight_layout()
plt.grid(True)
plt.show()

In [ ]:
country_year = (
    df.withColumn(
        "country",
        explode(split(col("country"), ","))
    )
    .groupBy("year_added", "country")
    .count()
)

windowSpec = Window.partitionBy(
    "year_added"
).orderBy(col("count").desc())

ranked_df = country_year.withColumn(
    "rank",
    rank().over(windowSpec)
)

top_country_each_year = ranked_df.filter(
    col("rank") == 1
)

top_country_each_year.show()

# Spark SQL Analytics

Spark SQL enables SQL-style querying on distributed datasets and simplifies analytical reporting workflows.

In [ ]:
df.createOrReplaceTempView("netflix")

In [ ]:
spark.sql("""
SELECT type, COUNT(*) as total
FROM netflix
GROUP BY type
""").show()

In [ ]:
spark.sql("""
SELECT rating, COUNT(*) as total
FROM netflix
GROUP BY rating
ORDER BY total DESC
""").show()

In [ ]:
final_pd = df.toPandas()

final_pd.to_csv(
    "final_netflix_analysis.csv",
    index=False
)

print("CSV exported successfully!")

# Conclusion

This project demonstrated scalable Netflix content analytics using PySpark and distributed computing concepts.

The workflow included:
- distributed preprocessing,
- feature engineering,
- Spark SQL analytics,
- partition optimization,
- caching,
- window functions,
- and visualization-driven business insights.

The project highlights how PySpark enables efficient large-scale analytics workflows suitable for real-world streaming platform data analysis.